# Pipeline v2 Apps Workbench

Temporary notebook runner for `src.pipeline_v2.apps.*` modules.

Each app stays pure: it accepts a dict and returns a dict. This notebook builds app inputs, calls app entrypoints, and writes notebook-run outputs under `output_v2/`.

In [1]:
from __future__ import annotations

from copy import deepcopy
from datetime import datetime, timedelta, timezone
import json
from pathlib import Path
import sys

import yaml

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "src").exists() and (PROJECT_ROOT.parent / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / "src" / "pipeline_v2" / "apps").exists():
    raise RuntimeError("Run this notebook from the repository root or output_v2 directory.")

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

APPS_DIR = PROJECT_ROOT / "src" / "pipeline_v2" / "apps"
NEWS_COLLECTION_DIR = APPS_DIR / "news_collection"
OUTPUT_V2 = PROJECT_ROOT / "output_v2"
IST = timezone(timedelta(hours=5, minutes=30))

print(f"Project root: {PROJECT_ROOT}")
print(f"Apps dir:     {APPS_DIR}")
print(f"Output root:  {OUTPUT_V2}")

Project root: /Users/manikantamb/_MyFiles/Work_/MyWork_Local/AI_Content_Creation
Apps dir:     /Users/manikantamb/_MyFiles/Work_/MyWork_Local/AI_Content_Creation/src/pipeline_v2/apps
Output root:  /Users/manikantamb/_MyFiles/Work_/MyWork_Local/AI_Content_Creation/output_v2


In [2]:
# Change these values for a different v2 app run.
RUN_PROFILE = "daily_48h_all_sources"
RUN_ID = f"v2_apps_{datetime.now(IST).strftime('%Y_%m_%d_%H%M%S')}_{RUN_PROFILE}"

NEWS_OUTPUT_DIR = OUTPUT_V2 / "news_collection" / RUN_ID
EVIDENCE_OUTPUT_DIR = OUTPUT_V2 / "evidence_prep" / RUN_ID
STORY_OUTPUT_DIR = OUTPUT_V2 / "story_mapping" / RUN_ID
VIDEO_OUTPUT_DIR = OUTPUT_V2 / "video_packaging" / RUN_ID
SCRIPT_OUTPUT_DIR = OUTPUT_V2 / "script_generation" / RUN_ID
METADATA_OUTPUT_DIR = OUTPUT_V2 / "metadata_creator" / RUN_ID

VOICEOVER_PROVIDER = "kokoro_local"  # kokoro_local | chatterbox_local | f5_tts_local | dia_transformers
VOICEOVER_STYLE = "general_youtube"
VOICEOVER_VARIANT = "kokoro_balanced_098"  # f"{VOICEOVER_PROVIDER}_{VOICEOVER_STYLE}"
VOICEOVER_SPEED = 0.98  # Use 0.85-0.95 for slower delivery, 0.98-1.02 for balanced pace. Chatterbox ignores this.

VOICEOVER_OUTPUT_DIR = OUTPUT_V2 / "voiceover" / f"{RUN_ID}_{VOICEOVER_VARIANT}"
VOICEOVER_MODEL_CACHE_ROOT = OUTPUT_V2 / "model_cache"
VOICEOVER_MODEL_CACHE_DIR = VOICEOVER_MODEL_CACHE_ROOT / "huggingface_local"
VOICEOVER_F5_REF_AUDIO_PATH = OUTPUT_V2 / "voiceover" / "_smoke_chatterbox_no_xet" / "audio" / "full_voiceover.wav"
VOICEOVER_F5_REF_TEXT = "Today, we are testing a more expressive local voiceover model for the new pipeline."
VOICEOVER_INPUT_DIR = OUTPUT_V2 / "voiceover_inputs"

print(f"Run profile: {RUN_PROFILE}")
print(f"Run ID:      {RUN_ID}")


Run profile: daily_48h_all_sources
Run ID:      v2_apps_2026_06_04_004549_daily_48h_all_sources


In [3]:
def load_yaml(path: Path) -> dict:
    data = yaml.safe_load(path.read_text(encoding="utf-8"))
    if not isinstance(data, dict):
        raise ValueError(f"Expected YAML object at {path}")
    return data


def write_json(path: Path, payload: dict) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(payload, indent=2, ensure_ascii=False), encoding="utf-8")


def write_jsonl(path: Path, rows: list[dict]) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    text = "\n".join(json.dumps(row, ensure_ascii=False) for row in rows)
    if text:
        text += "\n"
    path.write_text(text, encoding="utf-8")


def collection_window_for_profile(profile: dict) -> dict:
    end = datetime.now(IST)
    lookback_hours = int(profile["lookback_hours"])
    start = end - timedelta(hours=lookback_hours)
    return {
        "start": start.isoformat(),
        "end": end.isoformat(),
        "lookback_hours": lookback_hours,
    }


def build_news_collection_input(run_id: str, run_profile: str) -> dict:
    profile_config = load_yaml(NEWS_COLLECTION_DIR / "profiles.yaml")
    source_config = load_yaml(NEWS_COLLECTION_DIR / "source_config.yaml")
    profiles = profile_config["profiles"]
    if run_profile not in profiles:
        raise KeyError(f"Unknown run profile: {run_profile}")

    profile = profiles[run_profile]
    source_groups = source_config["layer_1"]["source_groups"]
    window = collection_window_for_profile(profile)
    collectors = {}
    for group_name in profile["enabled_source_groups"]:
        collector_config = deepcopy(source_groups[group_name])
        collector_config["source_group"] = group_name
        collector_config["collection_window"] = window
        collector_config["query_themes"] = list(profile["query_themes"])
        collector_config["runtime"] = deepcopy(profile.get("collector_caps", {}).get(group_name, {}))
        collectors[group_name] = collector_config

    return {
        "run": {
            "run_id": run_id,
            "generated_at": datetime.now(IST).isoformat(),
        },
        "config": {
            "enabled_source_groups": list(profile["enabled_source_groups"]),
            "query_themes": list(profile["query_themes"]),
            "collectors": collectors,
        },
        "input": {
            "run_profile": run_profile,
            "channel": deepcopy(profile_config["channel"]),
        },
    }


def build_evidence_prep_input(run_id: str, news_input: dict, news_result: dict) -> dict:
    first_collector = next(iter(news_input["config"]["collectors"].values()))
    return {
        "run": {
            "run_id": run_id,
            "generated_at": datetime.now(IST).isoformat(),
        },
        "config": {
            "collection_window": deepcopy(first_collector["collection_window"]),
            "query_themes": list(news_input["config"]["query_themes"]),
            "max_research_papers": 12,
            "filter_policy": {},
        },
        "input": {
            "news_collection": news_result,
        },
    }


def build_story_mapping_input(run_id: str, evidence_input: dict, evidence_result: dict) -> dict:
    return {
        "run": {
            "run_id": run_id,
            "generated_at": datetime.now(IST).isoformat(),
        },
        "config": {
            "embedding_model": "all-MiniLM-L6-v2",
            "collection_window": deepcopy(evidence_input["config"]["collection_window"]),
            "use_llm_story_labels": True,
            "clustering": {
                "max_pair_hours": None,
                "semantic_primary_threshold": 0.80,
                "semantic_supported_threshold": 0.78,
                "semantic_title_anchor_threshold": 0.74,
                "semantic_merge_groups": [
                    "ai_company_blogs",
                    "tech_news_apis",
                    "youtube_podcast_transcripts",
                ],
            },
            "story_label": {
                "provider": "openai",
                "model": "gpt-5-nano",
                "prompt_version": "v1",
                "worker_count": 10,
                "temperature": 0.15,
                "max_records_per_story": 6,
            },
        },
        "input": {
            "evidence_prep": evidence_result,
        },
    }


def build_video_packaging_input(run_id: str, story_mapping_result: dict) -> dict:
    return {
        "run": {
            "run_id": run_id,
            "generated_at": datetime.now(IST).isoformat(),
        },
        "config": {
            "target_format": "roundup",
            "roundup": {
                "candidate_limit": 20,
                "min_stories": 4,
                "max_stories": 7,
                "target_duration_seconds": 480,
                "intro_seconds": 35,
                "closing_seconds": 20,
                "role_weights": {
                    "lead": 1.35,
                    "major": 1.05,
                    "brief": 0.75,
                },
                "max_per_entity": 2,
                "max_per_product": 1,
                "max_per_event_family": 2,
                "allowed_priority_buckets": ["high", "medium"],
                "exclude_noise": True,
                "exclude_tutorials": True,
            },
        },
        "input": {
            "story_mapping": story_mapping_result,
        },
    }

def build_script_generation_input(run_id: str, video_packaging_result: dict) -> dict:
    return {
        "run": {
            "run_id": run_id,
            "generated_at": datetime.now(IST).isoformat(),
        },
        "config": {
            "provider": "openai",
            "model": "gpt-5-nano",
            "prompt_version": "v1",
            "temperature": 0.4,
            "words_per_minute": 150,
            "duration_tolerance_seconds": 45,
            "worker_count": 1,
            "script_style": {
                "audience": "AI-curious professionals",
                "tone": "clear, energetic, grounded",
                "avoid": [
                    "dry technical reporting",
                    "overhype",
                    "fake certainty",
                    "source-by-source summaries",
                ],
            },
            "source_policy": {
                "use_selected_stories_only": True,
                "include_full_records": True,
            },
            "reflection": {
                "enabled": True,
                "max_revision_passes": 1,
                "target_score": 8,
            },
        },
        "input": {
            "video_packaging": video_packaging_result,
        },
    }


def build_metadata_creator_input(run_id: str, script_generation_result: dict, video_packaging_result: dict) -> dict:
    profile_config = load_yaml(NEWS_COLLECTION_DIR / "profiles.yaml")
    channel = deepcopy(profile_config.get("channel", {}))
    return {
        "run": {
            "run_id": run_id,
            "generated_at": datetime.now(IST).isoformat(),
        },
        "config": {
            "provider": "openai",
            "model": "gpt-5-nano",
            "prompt_version": "v1",
            "temperature": 0.4,
            "channel": {
                "channel_name": channel.get("channel_name", "ManiVerse - AI"),
                "content_promise": channel.get("content_promise", ""),
            },
            "metadata_policy": {
                "title_option_count": 8,
                "thumbnail_text_option_count": 5,
                "max_tags": 20,
                "include_source_urls": True,
            },
            "memory_context": {
                "recent_title_angles": [],
            },
        },
        "input": {
            "script_generation": script_generation_result,
            "video_packaging": video_packaging_result,
        },
    }


def build_voiceover_input(run_id: str, script_generation_result: dict, provider: str = VOICEOVER_PROVIDER) -> dict:
    config = {
        "provider": provider,
        "style": VOICEOVER_STYLE,
        "voice": None,
        "speed": VOICEOVER_SPEED,
        "language_code": "a",
        "sample_rate_hz": 24000,
        "keep_chunks": True,
        "force": True,
        "output_dir": str(VOICEOVER_OUTPUT_DIR),
        "chunking": {
            "chunk_max_chars": None,
            "max_sentences_per_chunk": None,
            "chunk_pause_seconds": None,
            "paragraph_pause_seconds": None,
        },
    }

    if provider == "chatterbox_local":
        config["chunking"].update({
            "chunk_max_chars": 850,
            "max_sentences_per_chunk": 3,
        })
        config["chatterbox"] = {
            "device": "auto",
            "exaggeration": 0.65,
            "cfg_weight": 0.5,
            "audio_prompt_path": None,
            "cache_dir": str(VOICEOVER_MODEL_CACHE_ROOT / "huggingface_local"),
        }
    elif provider == "f5_tts_local":
        config["f5_tts"] = {
            "model": "F5TTS_v1_Base",
            "device": "cpu",
            "ref_audio_path": str(VOICEOVER_F5_REF_AUDIO_PATH),
            "ref_text": VOICEOVER_F5_REF_TEXT,
            "cache_dir": str(VOICEOVER_MODEL_CACHE_ROOT / "f5_tts"),
            "nfe_step": 16,
            "cfg_strength": 2.0,
            "sway_sampling_coef": -1.0,
            "target_rms": 0.1,
            "cross_fade_duration": 0.15,
            "remove_silence": False,
            "seed": 1234,
        }
    elif provider == "dia_transformers":
        config["sample_rate_hz"] = 44100
        config["dia"] = {
            "model_checkpoint": "nari-labs/Dia-1.6B-0626",
            "device": "cpu",
            "cache_dir": str(VOICEOVER_MODEL_CACHE_ROOT / "dia"),
            "max_new_tokens": 256,
            "guidance_scale": 3.0,
            "temperature": 1.8,
            "top_p": 0.9,
            "top_k": 45,
            "speaker_tag": "S1",
        }

    return {
        "run": {
            "run_id": run_id,
            "generated_at": datetime.now(IST).isoformat(),
        },
        "config": config,
        "input": {
            "script_generation": script_generation_result,
        },
    }


## App 1: News Collection

In [4]:
from src.pipeline_v2.apps.news_collection import run as run_news_collection

news_input = build_news_collection_input(RUN_ID, RUN_PROFILE)
news_result = run_news_collection(news_input)

write_json(NEWS_OUTPUT_DIR / "news_collection_result.json", news_result)
write_json(NEWS_OUTPUT_DIR / "raw_records.json", news_result["data"])
write_json(NEWS_OUTPUT_DIR / "summary.json", news_result["summary"])

print(f"News collection status: {news_result['status']}")
print(f"Raw records:            {news_result['data']['record_count']}")
print(f"Wrote:                  {NEWS_OUTPUT_DIR}")

News collection status: partial
Raw records:            164
Wrote:                  /Users/manikantamb/_MyFiles/Work_/MyWork_Local/AI_Content_Creation/output_v2/news_collection/v2_apps_2026_06_03_173708_daily_48h_all_sources


In [5]:
news_result["summary"]

{'run_id': 'v2_apps_2026_06_03_173708_daily_48h_all_sources',
 'generated_at': '2026-06-03T17:41:07.574581+05:30',
 'status': 'partial',
 'enabled_source_groups': ['ai_company_blogs',
  'tech_news_apis',
  'research_papers',
  'youtube_podcast_transcripts',
  'community_signals'],
 'enabled_source_group_count': 5,
 'successful_source_groups': ['ai_company_blogs',
  'tech_news_apis',
  'research_papers',
  'youtube_podcast_transcripts',
  'community_signals'],
 'successful_source_group_count': 5,
 'total_records': 164,
 'elapsed_seconds': 239.24,
 'groups': {'ai_company_blogs': {'collector': 'company_blogs',
   'status': 'partial',
   'fetched_count': 76,
   'collected_count': 66,
   'filtered_out_count': 10,
   'sources': {'openai_news': {'status': 'ok',
     'fetched_count': 4,
     'collected_count': 4,
     'filtered_out_count': 0,
     'notes': ['Article fetch failed for https://openai.com/index/codex-for-knowledge-work',
      'Content integrity summary_only: Codex is becoming a p

## App 2: Evidence Prep

In [6]:
from src.pipeline_v2.apps.evidence_prep import run as run_evidence_prep

evidence_input = build_evidence_prep_input(RUN_ID, news_input, news_result)
evidence_result = run_evidence_prep(evidence_input)

write_json(EVIDENCE_OUTPUT_DIR / "evidence_prep_result.json", evidence_result)
write_json(EVIDENCE_OUTPUT_DIR / "prepared_records.json", evidence_result["data"])
write_json(EVIDENCE_OUTPUT_DIR / "summary.json", evidence_result["summary"])

print(f"Evidence prep status: {evidence_result['status']}")
print(f"Prepared records:     {evidence_result['data']['record_count']}")
print(f"Wrote:                {EVIDENCE_OUTPUT_DIR}")

Evidence prep status: ok
Prepared records:     87
Wrote:                /Users/manikantamb/_MyFiles/Work_/MyWork_Local/AI_Content_Creation/output_v2/evidence_prep/v2_apps_2026_06_03_173708_daily_48h_all_sources


In [7]:
evidence_result["summary"]

{'run_id': 'v2_apps_2026_06_03_173708_daily_48h_all_sources',
 'generated_at': '2026-06-03T17:41:07.830803+05:30',
 'status': 'ok',
 'input_record_count': 164,
 'prepared_record_count': 87,
 'excluded_count': 77,
 'duplicate_count': 6,
 'content_status_counts': {'full': 87, 'partial': 0, 'title_only': 0},
 'exclude_reason_counts': {'official_content_too_short': 3,
  'outside_manifest_window': 64,
  'duplicate_url': 6,
  'off_topic_crypto': 4},
 'source_groups': {'ai_company_blogs': {'input': 66,
   'prepared': 23,
   'duplicates': 1,
   'excluded': 43},
  'tech_news_apis': {'input': 69,
   'prepared': 37,
   'duplicates': 5,
   'excluded': 32},
  'research_papers': {'input': 10,
   'prepared': 10,
   'duplicates': 0,
   'excluded': 0},
  'youtube_podcast_transcripts': {'input': 4,
   'prepared': 2,
   'duplicates': 0,
   'excluded': 2},
  'community_signals': {'input': 15,
   'prepared': 15,
   'duplicates': 0,
   'excluded': 0}},
 'excluded_records': [{'record_id': 'L1_00004',
   'sou

## App 3: Story Mapping

In [8]:
# import json

# RUN_ID = "v2_apps_2026_06_03_030142_daily_48h_all_sources"

# with open(f"../output_v2/evidence_prep/{RUN_ID}/evidence_prep_result.json", encoding="utf-8") as f:
#     evidence_prep_result = json.load(f)   

In [9]:
from src.pipeline_v2.apps.story_mapping import run as run_story_mapping

story_mapping_input = build_story_mapping_input(RUN_ID, evidence_input, evidence_result)
story_mapping_result = run_story_mapping(story_mapping_input)

write_json(STORY_OUTPUT_DIR / "story_mapping_result.json", story_mapping_result)
write_json(STORY_OUTPUT_DIR / "stories.json", story_mapping_result["data"])
write_json(STORY_OUTPUT_DIR / "summary.json", story_mapping_result["summary"])
write_jsonl(STORY_OUTPUT_DIR / "llm_calls.jsonl", story_mapping_result["llm_calls"])

print(f"Story mapping status: {story_mapping_result['status']}")
print(f"Stories:              {story_mapping_result['data']['story_count']}")
print(f"Wrote:                {STORY_OUTPUT_DIR}")

/Users/manikantamb/Library/Python/3.14/lib/python/site-packages/langchain_core/_api/deprecation.py:25: UserWarning: Core Pydantic V1 functionality isn't compatible with Python 3.14 or greater.
  from pydantic.v1.fields import FieldInfo as FieldInfoV1
/Users/manikantamb/Library/Python/3.14/lib/python/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 9148.95it/s]


Story mapping status: ok
Stories:              82
Wrote:                /Users/manikantamb/_MyFiles/Work_/MyWork_Local/AI_Content_Creation/output_v2/story_mapping/v2_apps_2026_06_03_173708_daily_48h_all_sources


In [10]:
story_mapping_result["summary"]

{'run_id': 'v2_apps_2026_06_03_173708_daily_48h_all_sources',
 'generated_at': '2026-06-03T17:44:34.247765+05:30',
 'status': 'ok',
 'input_record_count': 87,
 'story_count': 82,
 'single_record_story_count': 80,
 'multi_record_story_count': 2,
 'semantic_merge_count': 2,
 'dropped_singleton_count': 3,
 'effective_max_pair_hours': 48.0,
 'priority_bucket_counts': {'medium': 23, 'low': 59},
 'source_group_mix': {'tech_news_apis': 35,
  'ai_company_blogs': 23,
  'research_papers': 10,
  'community_signals': 12,
  'youtube_podcast_transcripts': 2},
 'llm_runtime': {'use_llm_story_labels': True,
  'story_label_provider': 'openai',
  'story_label_model': 'gpt-5-nano',
  'story_label_prompt_version': 'v1',
  'story_label_worker_count': 10,
  'llm_story_label_count': 82,
  'parallelized_steps': ['story_label'],
  'serial_steps': []},
 'top_story_quality_flags': {'low_editorial_value_count': 0,
  'tutorial_count': 0,
  'weak_content_count': 0,
  'event_update_count': 0},
 'top_stories': [{'ran

## App 4: Video Packaging

In [11]:
# import json

# RUN_ID = "v2_apps_2026_06_03_030142_daily_48h_all_sources"

# with open(f"../output_v2/story_mapping/{RUN_ID}/story_mapping_result.json", encoding="utf-8") as f:
#     story_mapping_result = json.load(f)   

In [12]:
from src.pipeline_v2.apps.video_packaging import run as run_video_packaging

video_packaging_input = build_video_packaging_input(RUN_ID, story_mapping_result)
video_packaging_result = run_video_packaging(video_packaging_input)

write_json(VIDEO_OUTPUT_DIR / "video_packaging_result.json", video_packaging_result)
write_json(VIDEO_OUTPUT_DIR / "video_packet.json", video_packaging_result["data"]["video_packet"])
write_json(VIDEO_OUTPUT_DIR / "summary.json", video_packaging_result["summary"])

print(f"Video packaging status: {video_packaging_result['status']}")
print(f"Selected format:        {video_packaging_result['data']['selected_format']}")
print(f"Selected stories:       {video_packaging_result['summary']['selected_story_count']}")
print(f"Ready for script gen:   {video_packaging_result['handoff']['ready_for_script_generation']}")
print(f"Wrote:                  {VIDEO_OUTPUT_DIR}")

Video packaging status: ok
Selected format:        roundup
Selected stories:       7
Ready for script gen:   True
Wrote:                  /Users/manikantamb/_MyFiles/Work_/MyWork_Local/AI_Content_Creation/output_v2/video_packaging/v2_apps_2026_06_03_173708_daily_48h_all_sources


In [13]:
video_packaging_result["summary"]

{'input_story_count': 82,
 'candidate_story_count': 20,
 'selected_story_count': 7,
 'rejected_story_count': 75,
 'selection_reason_counts': {'outside_candidate_limit': 62,
  'after_max_stories': 13},
 'role_counts': {'lead': 1, 'major': 2, 'brief': 4},
 'target_duration_seconds': 480,
 'failure_reason': '',
 'selected_story_ids': ['story_20260602_trump_signs_narrower_executive_order_on_60d9f7',
  'story_20260602_australia_gets_access_to_ai_model_too_d_5faeb1',
  'story_20260602_holo3_1_fast_local_computer_use_agent_2e6132',
  'story_20260602_mai_image_2_5_launches_at_no_2_for_imag_5ba5df',
  'story_20260602_how_baz_improved_its_ai_agent_code_revie_6044e7',
  'story_20260602_introducing_mai_voice_2_2b6c32',
  'story_20260602_amazon_faces_class_action_lawsuit_over_r_b73a25'],
 'rejected_story_ids': {'story_20260602_google_rolls_out_fake_call_detection_to_117114': ['after_max_stories'],
  'story_20260602_microsoft_offers_devs_a_better_way_to_co_e5baab': ['after_max_stories'],
  'story_20

## App 5: Script Generation


In [4]:
import json

RUN_ID = "v2_apps_2026_06_03_173708_daily_48h_all_sources"

with open(f"../output_v2/video_packaging/{RUN_ID}/video_packaging_result.json", encoding="utf-8") as f:
    video_packaging_result = json.load(f)


In [5]:
from src.pipeline_v2.apps.script_generation.formats.roundup import run as run_roundup_script_generation

script_generation_input = build_script_generation_input(RUN_ID, video_packaging_result)
script_generation_result = run_roundup_script_generation(script_generation_input)
script_package = script_generation_result["data"]["script_package"]

write_json(SCRIPT_OUTPUT_DIR / "script_generation_result.json", script_generation_result)
write_json(SCRIPT_OUTPUT_DIR / "script_package.json", script_package)
write_json(SCRIPT_OUTPUT_DIR / "summary.json", script_generation_result["summary"])
write_jsonl(SCRIPT_OUTPUT_DIR / "llm_calls.jsonl", script_generation_result["llm_calls"])
(SCRIPT_OUTPUT_DIR / "full_script.txt").parent.mkdir(parents=True, exist_ok=True)
(SCRIPT_OUTPUT_DIR / "full_script.txt").write_text(script_package["full_script"], encoding="utf-8")

print(f"Script generation status: {script_generation_result['status']}")
print(f"Sections:                 {script_generation_result['summary']['section_count']}")
print(f"Words:                    {script_generation_result['summary']['total_word_count']}")
print(f"Estimated seconds:        {script_generation_result['summary']['estimated_duration_seconds']}")
print(f"Ready for verification:   {script_generation_result['handoff']['ready_for_verification']}")
print(f"Wrote:                    {SCRIPT_OUTPUT_DIR}")


/Users/manikantamb/Library/Python/3.14/lib/python/site-packages/langchain_core/_api/deprecation.py:25: UserWarning: Core Pydantic V1 functionality isn't compatible with Python 3.14 or greater.
  from pydantic.v1.fields import FieldInfo as FieldInfoV1
/Users/manikantamb/Library/Python/3.14/lib/python/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Script generation status: failed
Sections:                 9
Words:                    1079
Estimated seconds:        431.6
Ready for verification:   False
Wrote:                    /Users/manikantamb/_MyFiles/Work_/MyWork_Local/AI_Content_Creation/output_v2/script_generation/v2_apps_2026_06_03_184426_daily_48h_all_sources


In [6]:
script_generation_result["summary"]


{'section_count': 9,
 'story_section_count': 7,
 'total_word_count': 1079,
 'estimated_duration_seconds': 431.6,
 'llm_call_count': 11,
 'failure_reason': 'Full script duration drift exceeds tolerance: estimated 431.6s, target 480s, tolerance 45s',
 'run_id': 'v2_apps_2026_06_03_173708_daily_48h_all_sources',
 'generated_at': '2026-06-03T18:44:36.034696+05:30',
 'status': 'failed',
 'selected_format': 'roundup',
 'llm_runtime': {'provider': 'openai',
  'model': 'gpt-5-nano',
  'prompt_version': 'v1',
  'worker_count': 1,
  'reflection_enabled': True,
  'max_revision_passes': 1,
  'target_score': 8,
  'parallelized_steps': [],
  'serial_steps': ['story_usage_analyzer',
   'roundup_full_script_writer',
   'roundup_script_reflector',
   'roundup_script_reviser']},
 'reflection': {'enabled': True,
  'iterations': [{'pass_index': 0,
    'draft_id': 'draft_00_initial',
    'reflection': {'overall_score': 7,
     'attention_score': 8,
     'clarity_score': 8,
     'flow_score': 7,
     'pacin

## App 6: Metadata Creator

In [ ]:
from src.pipeline_v2.apps.metadata_creator import run as run_metadata_creator

metadata_creator_input = build_metadata_creator_input(RUN_ID, script_generation_result, video_packaging_result)
metadata_creator_result = run_metadata_creator(metadata_creator_input)
metadata_package = metadata_creator_result["data"]["metadata_package"]
youtube_metadata = metadata_package["youtube_metadata"]
metadata_output_dir = OUTPUT_V2 / "metadata_creator" / RUN_ID

write_json(metadata_output_dir / "metadata_creator_result.json", metadata_creator_result)
write_json(metadata_output_dir / "metadata_package.json", metadata_package)
write_json(metadata_output_dir / "youtube_metadata.json", youtube_metadata)
write_json(metadata_output_dir / "summary.json", metadata_creator_result["summary"])
write_jsonl(metadata_output_dir / "llm_calls.jsonl", metadata_creator_result["llm_calls"])

print(f"Metadata status:          {metadata_creator_result['status']}")
print(f"Title options:            {metadata_creator_result['summary']['title_option_count']}")
print(f"Thumbnail options:        {metadata_creator_result['summary']['thumbnail_option_count']}")
print(f"Chapters:                 {metadata_creator_result['summary']['chapter_count']}")
print(f"Source URLs:              {metadata_creator_result['summary']['source_url_count']}")
print(f"Ready for verification:   {metadata_creator_result['handoff']['ready_for_verification']}")
print(f"Wrote:                    {metadata_output_dir}")


In [ ]:
metadata_creator_result["summary"]

## Voiceover Test

In [ ]:
# import json

# RUN_ID = "v2_apps_2026_06_03_184426_daily_48h_all_sources"

# with open(f"../output_v2/script_generation/{RUN_ID}/script_generation_result.json", encoding="utf-8") as f:
#     script_generation_result = json.load(f)


In [ ]:
import os
import subprocess

from src.pipeline_v2.voiceover.text_chunks import split_narration_into_chunks

voiceover_input = build_voiceover_input(RUN_ID, script_generation_result)
script_text_for_preview = voiceover_input["input"]["script_generation"]["data"]["script_package"]["full_script"]
style_config = voiceover_input["config"]["chunking"]
planned_chunks = split_narration_into_chunks(
    script_text_for_preview,
    max_chars=style_config["chunk_max_chars"] or 560,
    max_sentences_per_chunk=style_config["max_sentences_per_chunk"] or 2,
    chunk_pause_seconds=style_config["chunk_pause_seconds"] or 0.16,
    paragraph_pause_seconds=style_config["paragraph_pause_seconds"] or 0.46,
)
print(f"Voiceover provider:       {voiceover_input['config']['provider']}")
print(f"Voiceover style:          {voiceover_input['config']['style']}")
print(f"Planned voiceover chunks: {len(planned_chunks)}")

VOICEOVER_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
VOICEOVER_INPUT_DIR.mkdir(parents=True, exist_ok=True)
voiceover_input_path = VOICEOVER_INPUT_DIR / f"{RUN_ID}_voiceover_input.json"
write_json(voiceover_input_path, voiceover_input)

voiceover_python = PROJECT_ROOT / ".venv-voiceover" / "bin" / "python"
if not voiceover_python.exists():
    raise FileNotFoundError(f"Voiceover venv Python not found: {voiceover_python}")

env = {**os.environ, "PYTHONPATH": str(PROJECT_ROOT)}
subprocess.run(
    [
        str(voiceover_python),
        "-m",
        "src.pipeline_v2.voiceover.cli",
        "--input-json",
        str(voiceover_input_path),
    ],
    cwd=PROJECT_ROOT,
    env=env,
    check=True,
)

voiceover_summary = json.loads((VOICEOVER_OUTPUT_DIR / "summary.json").read_text(encoding="utf-8"))
voiceover_package = json.loads((VOICEOVER_OUTPUT_DIR / "voiceover_package.json").read_text(encoding="utf-8"))

print(f"Voiceover status: {voiceover_summary['status']}")
print(f"Voice:            {voiceover_summary['voice']}")
print(f"Duration seconds: {voiceover_summary['duration_seconds']}")
print(f"Chunks:           {voiceover_summary['chunk_count']}")
print(f"Audio path:       {voiceover_summary['audio_path']}")
print(f"Wrote:            {VOICEOVER_OUTPUT_DIR}")

Voiceover provider:       kokoro_local
Voiceover style:          general_youtube
Planned voiceover chunks: 30


/Users/manikantamb/_MyFiles/Work_/MyWork_Local/AI_Content_Creation/.venv-voiceover/lib/python3.12/site-packages/torch/nn/modules/rnn.py:123: UserWarning: dropout option adds dropout after all but last recurrent layer, so non-zero dropout expects num_layers greater than 1, but got dropout=0.2 and num_layers=1
  warnings.warn(
/Users/manikantamb/_MyFiles/Work_/MyWork_Local/AI_Content_Creation/.venv-voiceover/lib/python3.12/site-packages/torch/nn/utils/weight_norm.py:143: FutureWarning: `torch.nn.utils.weight_norm` is deprecated in favor of `torch.nn.utils.parametrizations.weight_norm`.
  WeightNorm.apply(module, name, dim)


{
  "provider": "kokoro_local",
  "voice": "af_bella",
  "style": "general_youtube",
  "audio_path": "output_v2/voiceover/v2_apps_2026_06_04_004549_daily_48h_all_sources_kokoro_balanced_098/audio/full_voiceover.wav",
  "duration_seconds": 493.14,
  "chunk_count": 30,
  "word_count": 1079,
  "run_id": "v2_apps_2026_06_03_184426_daily_48h_all_sources",
  "status": "ready",
  "ready_for_video_assembly": true,
  "requires_manual_audio_review": true
}
Voiceover status: ready
Voice:            af_bella
Duration seconds: 493.14
Chunks:           30
Audio path:       output_v2/voiceover/v2_apps_2026_06_04_004549_daily_48h_all_sources_kokoro_balanced_098/audio/full_voiceover.wav
Wrote:            /Users/manikantamb/_MyFiles/Work_/MyWork_Local/AI_Content_Creation/output_v2/voiceover/v2_apps_2026_06_04_004549_daily_48h_all_sources_kokoro_balanced_098


In [6]:
voiceover_summary

{'run_id': 'v2_apps_2026_06_03_184426_daily_48h_all_sources',
 'status': 'ready',
 'provider': 'kokoro_local',
 'voice': 'af_bella',
 'style': 'general_youtube',
 'audio_path': 'output_v2/voiceover/v2_apps_2026_06_04_004549_daily_48h_all_sources_kokoro_balanced_098/audio/full_voiceover.wav',
 'duration_seconds': 493.14,
 'chunk_count': 30,
 'word_count': 1079,
 'ready_for_video_assembly': True,
 'requires_manual_audio_review': True}